# 📦 NovaRetail+ — Customer Behavior Analysis

**NovaRetail+** is a Latin American e-commerce platform with millions of users.

The **Growth & Retention** team wants to answer one key question for the end of 2024:

> **What customer behavior factors are most strongly associated with annual revenue generated?**

This is a **correlational (exploratory) analysis** — correlation ≠ causation.

**Dataset:** `novaretail_comportamiento_clientes_2024.csv` — 15,000 records, 12 columns.

| Column | Description |
|--------|-------------|
| `id_cliente` | Unique customer identifier |
| `edad` | Customer age |
| `nivel_ingreso` | Estimated monthly income |
| `visitas_mes` | Monthly visits to the app or website |
| `compras_mes` | Monthly purchases |
| `gasto_publicidad_dirigida` | Ad spend allocated to the user |
| `satisfaccion` | Satisfaction score (1–5) |
| `miembro_premium` | Premium membership (1 = yes, 0 = no) |
| `abandono` | Churned (1 = yes, 0 = no) |
| `tipo_dispositivo` | Device type: mobile, desktop, or tablet |
| `region` | Geographic region: north, south, west, or east |
| `ingreso_anual` | **Target variable** — annual revenue generated by the customer |


---
## Section 1 — Load and Explore the Dataset

Validate that the dataset loads correctly, check data types, missing values, and general ranges before doing any analysis.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr, chi2_contingency

In [ ]:
# Load dataset
df = pd.read_csv('/datasets/novaretail_comportamiento_clientes_2024.csv')

# Inspect structure
df.info()

In [ ]:
# First look at the data
df.head(5)

---
## Section 2 — Prepare Data and Document Assumptions

### Data Exploration and Cleaning

#### Numerical Variables
- `edad`, `nivel_ingreso`, `visitas_mes`, `compras_mes`, `gasto_publicidad_dirigida`, `satisfaccion`, `ingreso_anual`
- Most types are correct. `edad` could be cast to integer for cleanliness.

#### Binary Variables
- `miembro_premium`, `abandono` — encoded as 0/1, no transformation needed.

#### Categorical Variables
- `id_cliente`, `tipo_dispositivo`, `region` — correctly defined, no transformation needed.


In [ ]:
# Optional: cast edad to integer
# df['edad'] = df['edad'].astype(int)

# Verify dtypes
df.info()

#### Explore Numerical Variables

In [ ]:
# Descriptive statistics
df.describe()

**Diagnosis — Numerical Variables:**

- `edad`: range 18–75, mean ≈ 38. Balanced distribution, no extreme outliers.
- `nivel_ingreso`: mean ≈ 30,000, std ≈ 9,800. High variability; some users with values near 75,000.
- `visitas_mes`: avg 10 visits/month, range 1–25. Moderate spread — different engagement levels.
- `compras_mes`: most users make 0–2 purchases/month, median = 1. Low conversion rate overall.
- `gasto_publicidad_dirigida`: avg ≈ 20, max ≈ 75. Right-skewed — a few users receive much higher ad spend.
- `satisfaccion`: concentrated between 3 and 4, avg ≈ 3.6. Generally positive but room to improve.
- `miembro_premium`: ~14% of users. Small but potentially strategic segment.
- `abandono`: ~15% churn rate. Relevant for retention analysis.
- `ingreso_anual`: high dispersion; median ≈ 30, max ≈ 245. Notable outliers that may influence aggregates.


#### Explore Binary Variables

In [ ]:
# Verify binary columns only contain 0 and 1
for col in ['miembro_premium', 'abandono']:
    print(f"Unique values in {col}:")
    print(df[col].value_counts())
    print("-" * 40)

**Diagnosis — Binary Variables:**

- `miembro_premium`: correctly coded 0/1. Majority are non-premium — confirms it's a minority strategic segment.
- `abandono`: correctly coded 0/1. Most users have not churned, but the ~15% who have make this a key retention variable.


#### Explore Categorical Variables

In [ ]:
# Unique value counts per categorical column
for col in ['id_cliente', 'tipo_dispositivo', 'region']:
    print(f"Unique values in {col}: {df[col].nunique()}")

In [ ]:
# Value distribution per categorical column
for col in ['tipo_dispositivo', 'region']:
    print(f"\nDistribution of {col}:")
    print(df[col].value_counts())
    print(df[col].value_counts(normalize=True).round(3))

**Diagnosis — Categorical Variables:**

- `id_cliente`: unique identifier per record. Not analytical — useful for traceability only.
- `tipo_dispositivo`: few categories (mobile, desktop, tablet). Relevant for usage pattern and churn analysis.
- `region`: geographic segmentation. Useful for regional comparisons and strategic targeting.


### Assumptions

- Analysis uses the **full dataset** — no filtering applied.
- Data has no nulls and types are adequately defined.
- Correlation method used depends on variable type:
  - **Pearson** — linear relationship between two numerical variables
  - **Spearman** — monotonic relationship; does not require normality
  - **Point-biserial** — numerical vs. binary variable
  - **Cramér's V** — association between two categorical variables
- **Central assumption:** this analysis identifies associations, not causal relationships.


---
## Section 3 — Visualizing Relationships

### Correlation Heatmap

In [ ]:
# Correlation matrix heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix — NovaRetail+")
plt.show()

**Observations (Heatmap):**

- Most variables show **low or near-zero correlations** — no dominant linear relationships across the board.
- `edad` and `nivel_ingreso` show near-zero correlations with everything — not key behavioral drivers.
- `abandono` has no strong correlations, suggesting its drivers may be non-linear or multi-variable.

**Regarding `ingreso_anual`:**
- **Very strong positive** correlation with `compras_mes` — the clearest signal in the data.
- **Moderate positive** with `visitas_mes` — more visits, more revenue, but weaker effect.
- **Weak positive** with `gasto_publicidad_dirigida` — advertising spend alone doesn't explain revenue.
- No meaningful correlation with `edad`, `nivel_ingreso`, `satisfaccion`, or `abandono`.


### Scatterplot — Key Pairs

A general scatterplot is not included — most variables show weak or null correlations that would only add visual noise. Instead, scatterplots are generated for the three most relevant pairs identified in the heatmap.


In [ ]:
# Monthly purchases vs Annual revenue
plt.figure(figsize=(6, 4))
sns.scatterplot(x='compras_mes', y='ingreso_anual', data=df)
plt.title('Monthly Purchases vs Annual Revenue')
plt.xlabel('Purchases per Month')
plt.ylabel('Annual Revenue')
plt.show()

In [ ]:
# Monthly visits vs Monthly purchases
plt.figure(figsize=(6, 4))
sns.scatterplot(x='visitas_mes', y='compras_mes', data=df)
plt.title('Monthly Visits vs Monthly Purchases')
plt.xlabel('Visits per Month')
plt.ylabel('Purchases per Month')
plt.show()

In [ ]:
# Monthly visits vs Directed ad spend
plt.figure(figsize=(6, 4))
sns.scatterplot(x='visitas_mes', y='gasto_publicidad_dirigida', data=df)
plt.title('Monthly Visits vs Directed Ad Spend')
plt.xlabel('Visits per Month')
plt.ylabel('Directed Ad Spend')
plt.show()

**Observations (Scatterplots):**

- **compras_mes vs ingreso_anual**: strong positive direction, low spread, near-linear — confirms very high correlation.
- **visitas_mes vs compras_mes**: positive direction, moderate spread — visits increase purchase likelihood but not proportionally.
- **visitas_mes vs gasto_publicidad_dirigida**: positive trend, medium-high spread — clear tendency but other factors also influence ad spend.


---
## Section 4 — Correlation Coefficients and Numerical Evidence

### Pearson and Spearman

In [ ]:
# Select relevant variables
relevant_vars = ['visitas_mes', 'compras_mes', 'gasto_publicidad_dirigida', 'ingreso_anual']

print("Pearson:")
print(df[relevant_vars].corr(method='pearson').round(2))

print("\nSpearman:")
print(df[relevant_vars].corr(method='spearman').round(2))

**Observations:**

- **compras_mes vs ingreso_anual**: Pearson = 0.97, Spearman = 0.97 — extremely strong and consistent across both methods.
- **visitas_mes vs gasto_publicidad_dirigida**: Pearson = 0.58, Spearman = 0.56 — moderate relationship, consistent.
- **visitas_mes vs compras_mes**: Pearson = 0.35, Spearman = 0.33 — positive but not strong. Visits increase purchase likelihood but not proportionally.


### Point-Biserial Correlation

In [ ]:
# Point-biserial: binary variables vs numerical variables
numeric_vars = ['visitas_mes', 'compras_mes', 'gasto_publicidad_dirigida', 'satisfaccion', 'ingreso_anual']

print("── abandono ──")
for var in numeric_vars:
    corr, p = pointbiserialr(df['abandono'], df[var])
    print(f"  vs {var}: r={corr:.3f}, p={p:.4f}")

print("\n── miembro_premium ──")
for var in numeric_vars:
    corr, p = pointbiserialr(df['miembro_premium'], df[var])
    print(f"  vs {var}: r={corr:.3f}, p={p:.4f}")

**Observations:**

**abandono:**
- All correlations very low (<0.1). Most are not statistically significant.
- Exception: `satisfaccion` — negative, weak, but significant (p<0.05). Higher satisfaction → lower churn.

**miembro_premium:**
- Most correlations very low and not significant.
- `satisfaccion`: positive, weak, significant — premium members tend to be more satisfied.
- `ingreso_anual`: positive, weak, significant — premium members generate slightly more revenue.


### Cramér's V — Categorical Association

In [ ]:
# Cramér's V function
def cramers_v(var1, var2, df):
    table = pd.crosstab(df[var1], df[var2])
    chi2, p, _, _ = chi2_contingency(table)
    n = table.sum().sum()
    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    return v, p

v, p = cramers_v('tipo_dispositivo', 'region', df)
print(f"tipo_dispositivo vs region → Cramér's V: {v:.3f}, p-value: {p:.4f}")

In [ ]:
# Contingency table
pd.crosstab(df['tipo_dispositivo'], df['region'])

**Observations:**

- Cramér's V = 0.012, p-value = 0.597 — virtually no association between device type and region.
- Device usage is independent of geography. No need to segment campaigns by this combination.


---
## Section 5 — Business Interpretation of Results

Each finding includes: visual evidence (if applicable), numerical evidence, interpretation, what we cannot claim, and the business implication.

---

### Finding 1 — Monthly Purchases and Annual Revenue

**Numerical evidence:** Pearson = 0.97, Spearman = 0.97

**Interpretation:** Very strong positive association — customers who buy more frequently generate significantly more annual revenue.

**We cannot claim:** That increasing purchases directly causes higher revenue, or that revenue depends solely on purchase count.

**Business implication:** High-frequency buyers are the most valuable segment. Retention and loyalty strategies should prioritize this group. Purchase frequency is a strong candidate variable for revenue prediction models.

---

### Finding 2 — Monthly Visits and Monthly Purchases

**Numerical evidence:** Pearson = 0.35, Spearman = 0.33

**Interpretation:** Moderate positive association — more visits tend to correlate with more purchases, but with notable variability between users.

**We cannot claim:** That more visits guarantee a purchase, or that driving traffic alone will increase conversions.

**Business implication:** Optimizing the user experience (reducing friction, improving navigation) may help convert visits into purchases. The funnel between visits and purchases should be analyzed.

---

### Finding 3 — Directed Ad Spend and Monthly Visits

**Numerical evidence:** Pearson = 0.58, Spearman = 0.56

**Interpretation:** Moderate-to-strong positive association — higher ad spend is associated with more visits to the platform.

**We cannot claim:** That ad spend is the only driver of visits, or that increasing budget will proportionally increase traffic.

**Business implication:** Directed advertising appears aligned with traffic growth. Evaluating ad ROI and optimizing campaigns before increasing budget is recommended.

---

### Finding 4 — Binary Variables (Churn and Premium Membership)

**Numerical evidence:** Point-biserial correlations < 0.1 for most variables. Exceptions: `satisfaccion` is weakly but significantly correlated with both `abandono` (negative) and `miembro_premium` (positive).

**Interpretation:** No strong linear relationship between churn or premium membership and the numerical variables analyzed. Satisfaction is the only meaningful (though weak) signal.

**We cannot claim:** That these variables are irrelevant in multivariate or non-linear models.

**Business implication:** Churn may depend on factors not captured here (support quality, pricing, UX). Qualitative or temporal variables should be included in future retention analysis.

---

### Finding 5 — Device Type and Region

**Numerical evidence:** Cramér's V = 0.012, p-value = 0.597

**Interpretation:** No statistically significant association between device type and geographic region.

**Business implication:** Segmentation strategies combining device and region are not justified by the data. Simplifying these segmentation rules is appropriate.


---
## Section 6 — Limitations and Next Steps

### Limitations

- **Correlation ≠ causation.** All findings describe associations, not causal relationships.
- **Bivariate analysis only.** Results reflect pairwise relationships — combined or interaction effects are not captured.
- **Unobserved variables.** Factors like pricing, promotions, customer support, or UX quality are not in the dataset and may influence behavior significantly.

### Next Steps

1. **Segmented analysis** — examine whether correlations hold across customer subgroups (e.g., premium vs. non-premium, high vs. low spenders)
2. **Multivariate modeling** — use regression to assess the relative weight of each variable on annual revenue
3. **Predictive modeling** — build models to predict annual revenue or churn probability; evaluate with RMSE, AUC, or accuracy
4. **Data enrichment** — integrate temporal variables, promotional history, and customer support records to deepen behavioral analysis
